# 05 — Evaluate frozen models and compare results

**Viraat Chaudhary · DATA 266 Lab 1 · Team 15 · Yelp Polarity**

Complete notebooks 02–04 first. Keep A100 for the first official-test inference. All three checkpoints are frozen before test text is opened. Reruns reuse saved predictions; test scores must not drive retraining, hyperparameter changes or checkpoint selection.

Run top-to-bottom. Save the executed notebook using **File → Download → Download .ipynb**. Use the exact filename provided. Outputs persist in your own Drive run folder. Source code is assistant-assisted; student explanations and analysis must be your own.


In [1]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import hashlib
import importlib.util
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path
from google.colab import drive, files

# Set these once; keep the same RUN_NAME for notebooks 02–07 and resumed sessions.
DRIVE_BASE_RELATIVE = 'DATA266_Lab1_Task2/Viraat_Chaudhary'
RUN_NAME = 'task2_models_rnn_lstm_tcn_v1'
PACKAGE_ZIP_NAME = 'Viraat_Task2_Run_Package.zip'
PACKAGE_ID = 'viraat_yelp_rnn_lstm_tcn_v1'

drive.mount('/content/drive', force_remount=False)
DRIVE_BASE = Path('/content/drive/MyDrive') / DRIVE_BASE_RELATIVE
ROOT = DRIVE_BASE / RUN_NAME / 'task2_sentiment/Viraat_Chaudhary'
CACHE_ROOT = Path('/content/viraat_task2_data')
if not ROOT.exists():
    archive = DRIVE_BASE / PACKAGE_ZIP_NAME
    if not archive.exists():
        print('Upload Viraat_Task2_Run_Package.zip (the code package, not a dataset ZIP).')
        uploaded = files.upload()
        if PACKAGE_ZIP_NAME not in uploaded:
            raise FileNotFoundError('Select the exact package ZIP filename.')
        archive = Path('/content') / PACKAGE_ZIP_NAME
        archive.write_bytes(uploaded[PACKAGE_ZIP_NAME])
    extraction = Path('/content/viraat_task2_package_extract')
    extraction.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as bundle:
        for info in bundle.infolist():
            if not (extraction / info.filename).resolve().is_relative_to(extraction.resolve()):
                raise ValueError('Invalid package ZIP path.')
        if bundle.testzip() is not None:
            raise ValueError('Package ZIP CRC failed.')
        bundle.extractall(extraction)
    manifest = json.loads((extraction / 'PACKAGE_CONTENT_MANIFEST.json').read_text())
    if manifest['package_id'] != PACKAGE_ID:
        raise ValueError('Incorrect package identity.')
    for item in manifest['files']:
        path = extraction / item['path']
        if hashlib.sha256(path.read_bytes()).hexdigest() != item['sha256']:
            raise ValueError('Package integrity failed: ' + item['path'])
    source = extraction / 'task2_sentiment/Viraat_Chaudhary'
    ROOT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, ROOT)
    (ROOT / 'package_installation.json').write_text(json.dumps({'package_id': PACKAGE_ID,
                                                              'package_manifest_sha256': hashlib.sha256(
        (extraction / 'PACKAGE_CONTENT_MANIFEST.json').read_bytes()).hexdigest()}, indent=2) + '\n')
else:
    installed = json.loads((ROOT / 'package_installation.json').read_text())
    if installed['package_id'] != PACKAGE_ID:
        raise ValueError('This RUN_NAME belongs to a different package.')

required = {'numpy': 'numpy==2.1.3', 'scipy': 'scipy>=1.14,<2',
            'sklearn': 'scikit-learn==1.6.1', 'nltk': 'nltk==3.9.1',
            'datasets': 'datasets==4.8.5', 'matplotlib': 'matplotlib==3.10.0',
            'pandas': 'pandas>=2.2,<4', 'tqdm': 'tqdm==4.67.3'}
missing = [spec for module, spec in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('Use a Colab runtime with PyTorch installed.')
# Keep the tokenizer version consistent with the approved snapshot.
import importlib.metadata
if importlib.metadata.version('nltk') != '3.9.1':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nltk==3.9.1'])

source_dir = str(ROOT / 'src')
sys.path.insert(0, source_dir)
for name in ('runtime_utils', 'yelp_data', 'data_runtime', 'models', 'metrics',
             'training', 'evaluation', 'review', 'reporting', 'evaluator'):
    module = sys.modules.get(name)
    if module is not None and Path(getattr(module, '__file__', '')).parent != ROOT / 'src':
        del sys.modules[name]
from data_runtime import verify_preprocessing
verify_preprocessing(ROOT)
import torch
import pandas as pd
print('Member folder:', ROOT)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Approved preprocessing restored; original notebook 01 is preserved.')


Mounted at /content/drive
Member folder: /content/drive/MyDrive/DATA266_Lab1_Task2/Viraat_Chaudhary/task2_models_rnn_lstm_tcn_v1/task2_sentiment/Viraat_Chaudhary
GPU: NVIDIA A100-SXM4-40GB
Approved preprocessing restored; original notebook 01 is preserved.


## Run the final evaluation

This computes every required classification/calibration metric, 2,000 shared bootstrap replicates for accuracy/macro-F1/MCC, paired McNemar tests with Holm adjustment, frozen slices and resources. PR-AUC is average precision; trapezoidal PR area is separately labelled. ECE uses predicted-class confidence and 15 uniform bins.

In [2]:
from evaluation import evaluate_models
FINAL_ROWS = evaluate_models(ROOT, DRIVE_BASE, CACHE_ROOT)
pd.DataFrame(FINAL_ROWS)[['model','accuracy','f1_macro','mcc','roc_auc','pr_auc_average_precision','brier_score','ece_15_bins']]


,model,accuracy,f1_macro,mcc,roc_auc,pr_auc_average_precision,brier_score,ece_15_bins
0,baseline,0.942658,0.942657,0.885343,0.986512,0.986900,0.042879,0.002350
1,experiment_1,0.948474,0.948472,0.897019,0.989121,0.989355,0.038735,0.008854
2,experiment_2,0.948026,0.948021,0.896242,0.989122,0.989492,0.039448,0.011682


## Own and team tables

All own metrics are in metrics_report.csv. Anshika values come from her uploaded executed evaluation output, not from rerunning or copying her models. Exact teammate hardware is not available in that upload. Per-slice memberships differ by tokenizer. Raw cross-member speed is not a controlled comparison.

In [3]:
from reporting import team_comparison, refresh_reports, export_artifacts
refresh_reports(ROOT)
TEAM_ROWS = team_comparison(ROOT)
display(pd.DataFrame(TEAM_ROWS)[['member','model','accuracy','f1_macro','mcc','roc_auc','pr_auc_average_precision']])
display(pd.read_csv(ROOT / 'outputs/metrics/mcnemar_tests.csv'))
display(pd.read_csv(ROOT / 'outputs/metrics/robustness_slice_metrics.csv'))
print('Evidence:', export_artifacts(ROOT, '05_Evaluation_Evidence'))


,member,model,accuracy,f1_macro,mcc,roc_auc,pr_auc_average_precision
0,Viraat_Chaudhary,baseline,0.9426578947368421,0.9426570131082743,0.8853430135768008,0.986512369806094,0.9869001206536869
1,Viraat_Chaudhary,experiment_1,0.9484736842105264,0.9484716287851925,0.8970189340052213,0.9891207991689751,0.9893550165963445
2,Viraat_Chaudhary,experiment_2,0.9480263157894737,0.9480208125829976,0.8962424279062094,0.9891215110803324,0.9894920304736363
3,Anshika_Goel,baseline_mean_pool,0.929474,0.929474,0.858947,0.977648,0.977576
4,Anshika_Goel,textcnn,0.941737,0.941736,0.883491,0.985671,0.986079
5,Anshika_Goel,bigru_attention,0.944868,0.944864,0.889869,0.988415,0.988652


,model_a,model_b,a_correct_b_wrong,a_wrong_b_correct,discordant_pairs,chi_square_statistic,raw_p_value,continuity_correction,holm_adjusted_p_value,reject_after_holm_at_0_05
0,baseline,experiment_1,601,822,1423,34.012649,5.475495e-09,True,1.095099e-08,True
1,baseline,experiment_2,530,734,1264,32.602057,1.130950e-08,True,1.130950e-08,True


,model,slice,count,fraction,macro_f1,error_rate
0,baseline,short_reviews,9752,0.256632,0.942021,0.055886
1,baseline,medium_reviews,18865,0.496447,0.943121,0.056878
2,baseline,long_reviews,9383,0.246921,0.937921,0.059789
3,baseline,contains_negation,28243,0.743237,0.937904,0.060227
4,baseline,high_oov_rate,2427,0.063868,0.922524,0.073754
5,experiment_1,short_reviews,9752,0.256632,0.942486,0.055681
6,experiment_1,medium_reviews,18865,0.496447,0.951814,0.048184
7,experiment_1,long_reviews,9383,0.246921,0.943835,0.053927
8,experiment_1,contains_negation,28243,0.743237,0.946238,0.051942
9,experiment_1,high_oov_rate,2427,0.063868,0.927306,0.069633


Evidence: /content/drive/MyDrive/DATA266_Lab1_Task2/Viraat_Chaudhary/task2_models_rnn_lstm_tcn_v1/task2_sentiment/Viraat_Chaudhary/exports/Viraat_Task2_05_Evaluation_Evidence.zip
